In [ ]:
"""
SEQUENTIAL CHAIN MULTI-AGENT PIPELINE on GSM8K (role-specialized agents)

Topology: a directed chain (optionally closing into a ring), NOT all-to-all.
Each agent has a FIXED ROLE and ASSUMES its input is correct, producing the NEXT
piece of information (it does not re-answer or correct the previous agent):

    Extractor  -> Formulator -> Solver -> Verifier
    (givens+goal) (equations)  (number)  (final answer / loop-back)

Aim: study how correct information survives (or fails to survive) transmission
through a dependency chain of specialized agents, and WHERE errors originate and
propagate. Because each stage depends on the last, an early error compounds -- and
because roles are fixed, we can locate WHICH stage broke. Correctness is checkable
against GSM8K ground truth at the final step (and inspectable at each hand-off).

Design choices (with known caveats, documented honestly):
  - fixed roles: realistic agentic design; caveat = our decomposition is part of
    what is being tested, so a failure may be the role-split, not the agent.
  - strict dependency (no correction): by design there is NO group error-correction;
    expect chains to be FRAGILE -- that fragility is itself the finding, not a bug.
  - capability-diverse: roles can be assigned to different models to study whether
    a weak agent at an early role poisons the whole chain.

RUN ON COLAB (GPU):
  !pip install transformers accelerate bitsandbytes datasets torch
  python chain_gsm8k_runner.py
"""

import json, re
import torch

# ------------------------------ CONFIG ------------------------------------
CONFIG = dict(
    n_problems=60,
    # assign a model to each ROLE (capability-diverse chain). Study knob: which
    # role gets the weaker model changes where the chain is likely to break.
    role_models=dict(extractor="qwen", formulator="qwen", solver="mistral", verifier="qwen"),
    qwen_model="Qwen/Qwen2.5-7B-Instruct",
    mistral_model="mistralai/Mistral-7B-Instruct-v0.2",
    max_new_tokens=512,
    close_loop=False,       # if True, Verifier feeds back to Extractor for a consistency check
    seed=0,
    out_path="chain_gsm8k_transcripts.json",
    save_to_drive=True,
    drive_dir="/content/drive/MyDrive/holonomy_debate",
)
# ---------------------------------------------------------------------------

_MODELS, _TOKENIZERS = {}, {}


def load_models(cfg):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_type="nf4")
    needed = set(cfg["role_models"].values())
    name_of = dict(qwen=cfg["qwen_model"], mistral=cfg["mistral_model"])
    for key in needed:
        print(f"loading {key} = {name_of[key]} (4-bit) ...")
        _TOKENIZERS[key] = AutoTokenizer.from_pretrained(name_of[key])
        _MODELS[key] = AutoModelForCausalLM.from_pretrained(
            name_of[key], quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
        _MODELS[key].eval()


def generate(prompt, model_key, cfg):
    tok, mdl = _TOKENIZERS[model_key], _MODELS[model_key]
    msgs = [{"role": "system", "content": "You are a careful math assistant. Follow the role instructions exactly."},
            {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=cfg["max_new_tokens"],
                           do_sample=True, temperature=0.7, top_p=0.9,
                           pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


# ---- role prompts: each consumes the previous role's output as GIVEN-correct ----

def extractor_prompt(problem):
    return (f"You are the EXTRACTOR. Read the math word problem and list ONLY:\n"
            f"(1) the known quantities (with their meaning), and (2) exactly what is being asked.\n"
            f"Do NOT solve it. Do NOT set up equations.\n\n"
            f"Problem: {problem}\n\n"
            f"Respond as:\nGIVENS: <list>\nGOAL: <what to find>")

def formulator_prompt(problem, extraction):
    return (f"You are the FORMULATOR. Assume the extraction below is CORRECT.\n"
            f"Using it, write the arithmetic/equations needed to reach the goal.\n"
            f"Do NOT compute the final number; just set up the steps.\n\n"
            f"Problem: {problem}\n\nExtraction (assume correct):\n{extraction}\n\n"
            f"Respond as:\nEQUATIONS: <ordered steps/equations to compute>")

def solver_prompt(problem, formulation):
    return (f"You are the SOLVER. Assume the equations below are CORRECT.\n"
            f"Compute the numerical result by following them step by step.\n\n"
            f"Problem: {problem}\n\nEquations (assume correct):\n{formulation}\n\n"
            f"Respond as:\nWORK: <the arithmetic>\nRESULT: <final number>")

def verifier_prompt(problem, solution):
    return (f"You are the VERIFIER. You are given a proposed solution.\n"
            f"Sanity-check it against the problem and state the final answer.\n\n"
            f"Problem: {problem}\n\nProposed solution:\n{solution}\n\n"
            f"Respond as:\nCHECK: <brief sanity check>\nANSWER: <final number>")


def extract_number(text):
    """Pull the final numeric answer. Prefer an ANSWER:/RESULT: line; else last number."""
    for tag in ["ANSWER", "RESULT"]:
        m = re.findall(tag + r"\s*:?\s*\$?(-?[\d,]+(?:\.\d+)?)", text.upper())
        if m:
            return _to_num(m[-1])
    nums = re.findall(r"(-?[\d,]+(?:\.\d+)?)", text)
    return _to_num(nums[-1]) if nums else None

def _to_num(s):
    try:
        return float(s.replace(",", ""))
    except Exception:
        return None

def gsm8k_answer(ans_field):
    # GSM8K answers look like "...#### 42"
    m = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", ans_field)
    return _to_num(m.group(1)) if m else None


def run_chain(problem, cfg):
    rm = cfg["role_models"]
    ex = generate(extractor_prompt(problem), rm["extractor"], cfg)
    fo = generate(formulator_prompt(problem, ex), rm["formulator"], cfg)
    so = generate(solver_prompt(problem, fo), rm["solver"], cfg)
    ve = generate(verifier_prompt(problem, so), rm["verifier"], cfg)
    stages = dict(extraction=ex, formulation=fo, solution=so, verification=ve)

    loopback = None
    if cfg["close_loop"]:
        # Verifier's answer fed BACK to a fresh Extractor-style consistency check:
        # "does the final answer, read back into the problem, remain consistent?"
        lb_prompt = (f"You are the LOOP-CHECK. Given the problem and a final answer, state whether "
                     f"the answer is consistent with the problem's constraints.\n\n"
                     f"Problem: {problem}\n\nFinal answer: {extract_number(ve)}\n\n"
                     f"Respond as:\nCONSISTENT: <yes/no>\nWHY: <one line>")
        loopback = generate(lb_prompt, rm["extractor"], cfg)
        stages["loopback"] = loopback

    final = extract_number(ve)
    return stages, final


def main():
    cfg = CONFIG
    torch.manual_seed(cfg["seed"])
    from datasets import load_dataset
    print("loading GSM8K ...")
    ds = load_dataset("openai/gsm8k", "main", split="test")

    load_models(cfg)

    records, n_correct = [], 0
    for i in range(cfg["n_problems"]):
        problem = ds[i]["question"]
        gold = gsm8k_answer(ds[i]["answer"])
        stages, final = run_chain(problem, cfg)
        correct = (final is not None and gold is not None and abs(final - gold) < 1e-3)
        n_correct += int(correct)
        records.append(dict(problem=problem, gold=gold, final=final,
                            correct=correct, stages=stages))
        print(f"P{i:02d}: gold={gold}  chain_final={final}  {'OK' if correct else 'WRONG'}")

    import os
    out_path = cfg["out_path"]
    if cfg.get("save_to_drive"):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            os.makedirs(cfg["drive_dir"], exist_ok=True)
            out_path = os.path.join(cfg["drive_dir"], cfg["out_path"])
        except Exception as e:
            print(f"(Drive mount failed: {e}; saving locally)")
    with open(out_path, "w") as f:
        json.dump(records, f)

    print("\n===== CHAIN PIPELINE REPORT (GSM8K) =====")
    print(f"problems run:        {cfg['n_problems']}")
    print(f"chain got correct:   {n_correct}  ({100*n_correct/cfg['n_problems']:.0f}%)")
    print(f"role->model mapping: {cfg['role_models']}")
    print(f"transcripts saved:   {out_path}")
    print()
    print("Next: analyze WHERE chains break -- inspect stages of WRONG cases to see whether")
    print("the error originated at extraction, formulation, or solving, and how it propagated.")


if __name__ == "__main__":
    main()

loading GSM8K ...


loading qwen = Qwen/Qwen2.5-7B-Instruct (4-bit) ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


P00: gold=18.0  chain_final=18.0  OK
P01: gold=3.0  chain_final=3.0  OK
P02: gold=70000.0  chain_final=95000.0  WRONG
P03: gold=540.0  chain_final=540.0  OK
P04: gold=20.0  chain_final=20.0  OK
P05: gold=64.0  chain_final=64.0  OK
P06: gold=260.0  chain_final=240.0  WRONG
P07: gold=160.0  chain_final=420.0  WRONG
P08: gold=45.0  chain_final=135.0  WRONG
P09: gold=460.0  chain_final=460.0  OK
P10: gold=366.0  chain_final=366.0  OK
P11: gold=694.0  chain_final=694.0  OK
P12: gold=13.0  chain_final=12.0  WRONG
P13: gold=18.0  chain_final=None  WRONG
P14: gold=60.0  chain_final=60.0  OK
P15: gold=125.0  chain_final=125.0  OK
P16: gold=230.0  chain_final=230.0  OK
P17: gold=57500.0  chain_final=57500.0  OK
P18: gold=7.0  chain_final=7.0  OK
P19: gold=6.0  chain_final=6.0  OK
P20: gold=15.0  chain_final=11.0  WRONG
P21: gold=14.0  chain_final=2.0  WRONG
P22: gold=7.0  chain_final=7.0  OK
P23: gold=8.0  chain_final=8.0  OK
P24: gold=26.0  chain_final=26.67  WRONG
P25: gold=2.0  chain_final=No

In [ ]:
import json
recs = json.load(open("/content/drive/MyDrive/holonomy_debate/chain_gsm8k_transcripts.json"))
for i, r in enumerate(recs):
    if not r["correct"]:
        print("="*70)
        print(f"P{i:02d}  gold={r['gold']}  chain_final={r['final']}")
        print("-"*70)
        print("PROBLEM:", r["problem"])
        for stage in ["extraction","formulation","solution","verification"]:
            print(f"\n--- {stage.upper()} ---")
            print(r["stages"][stage])
        print()

P02  gold=70000.0  chain_final=95000.0
----------------------------------------------------------------------
PROBLEM: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

--- EXTRACTION ---
GIVENS: Josh buys a house for $80,000, puts in $50,000 in repairs.
GOAL: How much profit did he make?

--- FORMULATION ---
EQUATIONS: 
1. Total Investment = Purchase Price + Repairs
2. Increased Value = Total Investment * 1.5
3. Profit = Increased Value - Total Investment

--- SOLUTION ---
WORK: 
Total Investment = $80,000 + $50,000 = $130,000
Increased Value = $130,000 * 1.5 = $195,000
Profit = $195,000 - $130,000 = $65,000

RESULT: $65,000

--- VERIFICATION ---
CHECK: The calculation seems correct. The increased value is indeed 150% of the total investment, which is $130,000, resulting in an increase of $95,000, making the new value $225,000. The profit is then the sel